# 03 — Khai phá luật kết hợp Agent theo bản đồ

Mục tiêu: tìm các Agent thường xuất hiện cùng nhau trong một đội hình 5 người, và kiểm tra (một cách thận trọng) tỷ lệ thắng của các tổ hợp.

Đơn vị phân tích: **một đội trong một ván**, tập 5 Agent không có thứ tự.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from valorant_mining.config import DATA_PROCESSED, VALID_MAPS
from valorant_mining.rules import mine_map_rules, run_rules
from valorant_mining.plotting import plot_rules, plot_composition_volume

comps = pd.read_csv(DATA_PROCESSED / "team_compositions.csv")
comps.groupby("Map").agg(teams=("GameID", "size"), labeled=("won", "count"))


## 1. Một bản đồ cụ thể: Ascent

In [ ]:
rules_ascent, itemsets_ascent, baskets = mine_map_rules(comps, "Ascent", min_count=12)
rules_ascent[["rule", "support", "confidence", "lift", "support_count",
              "win_rate_with_antecedent", "win_rate_without_antecedent",
              "fisher_p_value"]].head(15)


## 2. Chạy cho tất cả bản đồ

In [ ]:
result = run_rules(comps)
print(result["summary"])
display(result["report_rules"])


## 3. Đội hình 5 Agent và tỷ lệ thắng

Khoảng tin cậy Wilson 95% được tính cho từng đội hình. Với mỗi đội hình, win rate được so sánh với phần còn lại của cùng map bằng Fisher exact; cột `fisher_q_bh` là p-value đã hiệu chỉnh Benjamini–Hochberg trên toàn bộ đội hình. Đây vẫn là phân tích khám phá, chưa kiểm soát đội/patch và chưa có hold-out theo thời gian.


In [ ]:
comps_top = result["compositions"].sort_values("n", ascending=False).head(15)
comps_top[["map", "agents_str", "n", "win_rate", "win_rate_ci_low", "win_rate_ci_high",
           "fisher_p_value", "fisher_q_bh", "significant_fdr"]]


In [ ]:
plot_rules(result)
plot_composition_volume(comps)
for name in ["03_agent_pickrate_map.png", "03_rules_top_lift.png", "03_compositions_winrate.png"]:
    display(Image(filename=str(ROOT / "reports" / "figures" / name)))


## 4. Kết luận cần nhớ

- Mỗi map có một “meta” riêng: Icebox gần như cố định Viper–Sage–Sova; Bind thiên về Brimstone–Viper–Skye; Haven là Cypher–Omen–Sova.
- Support/confidence/lift đo **đồng xuất hiện**, không đo nhân quả.
- Phần lớn luật có win rate quanh 50%. Sau hiệu chỉnh BH, chỉ một phần nhỏ luật và đội hình còn khác biệt có ý nghĩa; đây là tín hiệu khám phá, không phải kết luận nhân quả.
- Cần hold-out theo thời gian trước khi kết luận một tổ hợp “mạnh hơn”.
